# Building Neural Networks with PyTorch

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Understanding nn.Module

In [2]:
# Simple linear layer from scratch
# This handles all the gradient accumulation and computation for our layers automatically
class SimpleLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super(SimpleLinear, self).__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Initialize parameters
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.bias = nn.Parameter(torch.randn(out_features))
    
    def forward(self, x):
        return x @ self.weight.T + self.bias

# Test our custom layer, which we can use by calling it.
layer = SimpleLinear(4, 2)
x = torch.randn(50, 4)  # Batch of 3 (three samples with 4 dimensions)
output = layer(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Weight shape: {layer.weight.shape}")
print(f"Bias shape: {layer.bias.shape}")

Input shape: torch.Size([50, 4])
Output shape: torch.Size([50, 2])
Weight shape: torch.Size([2, 4])
Bias shape: torch.Size([2])


## 2. Built-in Layers and Components

In [3]:
# Common layers that torch handles for you. These cover most of the ones we discussed in class 

# Linear layers, this could be your input layer for example
linear = nn.Linear(10, 5)
print(f"Linear layer: {linear}")

# Activation functions
relu = nn.ReLU()
sigmoid = nn.Sigmoid()
tanh = nn.Tanh()

# Normalization layers
batch_norm = nn.BatchNorm1d(10)
layer_norm = nn.LayerNorm(10)

# Regularization
dropout = nn.Dropout(0.5)

# Convolutional layers, these are the two operations that we discussed in class. 
# the conv2d could be a decoder channel, where we goe from a compressed representation to a higher dimension
conv2d = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)

# Test with sample data, that could a 32by32 rgb image (the same dimensions you could have in your sentinel2 task)
x = torch.randn(1, 3, 32, 32)  # Batch=1, Channels=3, Height=32, Width=32
x = conv2d(x)
print(f"After conv2d: {x.shape}")
x = maxpool(x)
print(f"After maxpool: {x.shape}")

Linear layer: Linear(in_features=10, out_features=5, bias=True)
After conv2d: torch.Size([1, 16, 32, 32])
After maxpool: torch.Size([1, 16, 16, 16])


## 3. Building Complete Networks

In [4]:
# Multi-layer Perceptron (MLP), that we can flexibly define, with our chosen amount of layers
class MLP(nn.Module):
    def __init__(self, input_size, hidden_sizes, output_size, dropout_prob=0.5):
        super(MLP, self).__init__()
        
        # Build layers dynamically
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_prob)
            ])
            prev_size = hidden_size
        
        # Output layer with no activation, we will have to apply this later
        layers.append(nn.Linear(prev_size, output_size))
        
        # we add all the layers to a sequential torch module to make the forward pass easier.
        # we can simply not do this, but then we have to handle the forward computation in the forward for each layer that we have.
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Create different sized networks
small_mlp = MLP(input_size=784, hidden_sizes=[64], output_size=10)
medium_mlp = MLP(input_size=784, hidden_sizes=[128, 64], output_size=10)
large_mlp = MLP(input_size=784, hidden_sizes=[512, 256, 128], output_size=10)

# Test forward pass, imagine that we have 32 MNIST images, where we take all the samples 
x = torch.randn(32, 784)
output = small_mlp(x)
print(f"\nInput shape: {x.shape}")
print(f"Output shape: {output.shape}")


Input shape: torch.Size([32, 784])
Output shape: torch.Size([32, 10])


In [5]:
# Convolutional Neural Network
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        # Pooling (making the image smaller)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)
        
        # Dropout
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        # Conv block 1
        print(x.shape)
        x = self.pool(F.relu(self.conv1(x)))
        
        # Conv block 2
        print(x.shape)
        x = self.pool(F.relu(self.conv2(x)))
        
        # Conv block 3
        print(x.shape)
        x = self.pool(F.relu(self.conv3(x)))
        
        # Flatten for FC layers
        print(x.shape)
        # this says the shape of the tensor should be: xs first dimension and then whatever is required to squeeze the rest in
        x = x.view(x.size(0), -1)
        
        # FC layers
        print(x.shape)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# Create CNN
cnn = SimpleCNN()

# Test with MNIST-sized input
x = torch.randn(1, 1, 28, 28)
output = cnn(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

torch.Size([1, 1, 28, 28])
torch.Size([1, 32, 14, 14])
torch.Size([1, 64, 7, 7])
torch.Size([1, 128, 3, 3])
torch.Size([1, 1152])
Input shape: torch.Size([1, 1, 28, 28])
Output shape: torch.Size([1, 10])


## 4. Different Network Architectures for MNIST

In [6]:
# 1. Very simple model that will underfit
class TooSimpleNet(nn.Module):
    def __init__(self):
        super(TooSimpleNet, self).__init__()
        self.fc = nn.Linear(784, 10) 
    
    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten
        return self.fc(x)

# 2. Medium complexity model, this one will be probably work best
class GoodNet(nn.Module):
    def __init__(self):
        super(GoodNet, self).__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return self.fc3(x)

# 3. Overly complex model (overfitting candidate)
class TooComplexNet(nn.Module):
    def __init__(self):
        super(TooComplexNet, self).__init__()
        self.fc1 = nn.Linear(784, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256, 128)
        self.fc5 = nn.Linear(128, 64)
        self.fc6 = nn.Linear(64, 10)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))
        x = F.relu(self.fc5(x))
        return self.fc6(x)

# Create instances
too_simple = TooSimpleNet()
good_net = GoodNet()
too_complex = TooComplexNet()

# Compare parameter counts
models = {
    'Too Simple': too_simple,
    'Good Net': good_net,
    'Too Complex': too_complex
}

# Test forward pass, with 5 simulate mnist images
x_test = torch.randn(22, 1, 28, 28)
print(f"\nTesting with input shape: {x_test.shape}")
for name, model in models.items():
    output = model(x_test)
    print(f"{name:12}: output shape {output.shape}")


Testing with input shape: torch.Size([22, 1, 28, 28])
Too Simple  : output shape torch.Size([22, 10])
Good Net    : output shape torch.Size([22, 10])
Too Complex : output shape torch.Size([22, 10])


In [7]:
next(too_simple.parameters()).device

device(type='cpu')

In [8]:
# we have but one model and it is on the gpu
print(next(too_simple.parameters()).device)
way_too_simple = too_simple.to("cuda:0")
print(next(iter(too_simple.parameters())).device)
print(next(iter(way_too_simple.parameters())).device)

cpu
cuda:0
cuda:0
